In [2]:
from pathlib import Path

import pandas as pd
from nltk import Tree

PROJECT_ROOT = Path("..").resolve()
df = pd.read_feather(
    PROJECT_ROOT / "data" / "processed" / "constituency_features.feather"
)

## Sentence Type Distribution by Source and Domain

We first examine the overall distribution of Feng et al.'s (2012) 
sentence types across human, GPT, and Claude text, faceted by domain. 
This gives a descriptive baseline of what each source tends to produce 
before testing the regularity hypothesis (RQ1) at the document level.

Two classification schemes are examined:
- `sent_type_1`: SIMPLE / COMPOUND / COMPLEX / COMPLEX-COMPOUND / OTHER 
  (Algorithm 1 — based on top-level S/VP and SBAR presence)
- `sent_type_2`: PERIODIC / LOOSE / OTHER 
  (Algorithm 2 — based on position of main clause relative to 
  subordinate material)

In [3]:
feng_1_dist = (
    df.groupby(["domain", "source"])["feng_algo_1"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(3)
)
feng_1_dist

feng_algo_1    COMPLEX  COMPLEX-COMPOUND  COMPOUND  OTHER  SIMPLE
domain source                                                    
essay  claude    0.412             0.035     0.053  0.013   0.487
       gpt       0.306             0.028     0.067  0.057   0.542
       human     0.428             0.051     0.068  0.047   0.405
reuter claude    0.379             0.053     0.082  0.023   0.464
       gpt       0.452             0.064     0.065  0.021   0.398
       human     0.398             0.155     0.146  0.016   0.284
wp     claude    0.258             0.034     0.092  0.083   0.533
       gpt       0.379             0.059     0.126  0.032   0.403
       human     0.233             0.085     0.109  0.151   0.421

In [4]:
feng_2_dist = (
    df.groupby(["domain", "source"])["feng_algo_2"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(3)
)
feng_2_dist

feng_algo_2    LOOSE  OTHER  PERIODIC
domain source                        
essay  claude  0.503  0.278     0.219
       gpt     0.471  0.283     0.246
       human   0.489  0.276     0.234
reuter claude  0.524  0.259     0.216
       gpt     0.572  0.193     0.236
       human   0.433  0.193     0.374
wp     claude  0.373  0.418     0.209
       gpt     0.459  0.207     0.334
       human   0.292  0.438     0.269

In [7]:
doc_type1_props = (
    df.groupby(["domain", "source", "doc_id"])["feng_algo_1"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
doc_type1_props.head()

type1_variance = (
    doc_type1_props.reset_index()
    .groupby(["domain", "source"])
    [["COMPLEX", "COMPLEX-COMPOUND", "COMPOUND", "OTHER", "SIMPLE"]]
    .std()
    .round(3)
)
type1_variance

feng_algo_1    COMPLEX  COMPLEX-COMPOUND  COMPOUND  OTHER  SIMPLE
domain source                                                    
essay  claude    0.153             0.049     0.059  0.037   0.157
       gpt       0.141             0.037     0.069  0.092   0.149
       human     0.162             0.062     0.069  0.061   0.154
reuter claude    0.017             0.007     0.009  0.005   0.023
       gpt       0.026             0.010     0.009  0.005   0.020
       human     0.017             0.015     0.017  0.004   0.016
wp     claude    0.119             0.052     0.088  0.067   0.128
       gpt       0.119             0.054     0.088  0.051   0.119
       human     0.121             0.095     0.073  0.101   0.141

In [8]:
doc_type2_props = (
    df.groupby(["domain", "source", "doc_id"])["feng_algo_2"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)

type2_variance = (
    doc_type2_props.reset_index()
    .groupby(["domain", "source"])
    [["LOOSE", "OTHER", "PERIODIC"]]
    .std()
    .round(3)
)
type2_variance

feng_algo_2    LOOSE  OTHER  PERIODIC
domain source                        
essay  claude  0.135  0.134     0.119
       gpt     0.134  0.131     0.129
       human   0.147  0.141     0.132
reuter claude  0.018  0.019     0.015
       gpt     0.022  0.013     0.015
       human   0.017  0.014     0.019
wp     claude  0.121  0.137     0.109
       gpt     0.120  0.100     0.116
       human   0.110  0.163     0.142

In [9]:
sent_counts = df.groupby(["domain", "source", "doc_id"]).size().reset_index(name="n_sents") 
sent_counts.groupby(["domain", "source"])["n_sents"].describe().round(2)

count     mean    std     min      25%     50%      75%  \
domain source                                                             
essay  claude  1000.0    22.65   4.66    10.0    19.00    22.0    25.00   
       gpt      998.0    28.79   9.68     7.0    21.00    28.0    34.00   
       human    993.0    31.59  29.91     1.0    15.00    22.0    36.00   
reuter claude    20.0   905.90  23.43   870.0   888.25   907.0   924.25   
       gpt       20.0  1093.45  38.00  1022.0  1065.50  1097.0  1118.00   
       human     20.0  1024.60  45.93   950.0  1002.25  1021.5  1050.25   
wp     claude  1000.0    33.50  11.80     1.0    25.00    32.0    40.00   
       gpt     1000.0    31.28  13.81     1.0    21.00    32.0    41.00   
       human   1000.0    45.04  32.08     2.0    22.00    38.0    60.25   

                  max  
domain source          
essay  claude    45.0  
       gpt       80.0  
       human    365.0  
reuter claude   953.0  
       gpt     1157.0  
       human   1101.0  
wp     claude   100.0  
       gpt      143.0  
       human    235.0